## Objective
Build an AI-powered brochure generator for MapleStory jobs.

Given:
- A MapleStory resource website
- A target job (e.g. Night Lord, Kain, Mercedes)

The system will:

1. Scrape the website
2. Extract links
3. Use GPT to identify relevant job resources
4. Gather content from those resources
5. Generate a brochure for prospective players

This follows the same architecture as the Day 5 Company Brochure Generator.

## Imports

In [14]:
import os
import json
import requests

from dotenv import load_dotenv
from bs4 import BeautifulSoup

from IPython.display import Markdown
from IPython.display import display
from IPython.display import update_display

from openai import OpenAI

## OpenAI Setup

In [2]:
load_dotenv(override=True)

client = OpenAI()

MODEL = "gpt-4.1-mini"

## Website Scraper

In [ ]:
def fetch_website_contents(url):

    response = requests.get(
        url,
        headers={"User-Agent": "Mozilla/5.0"},
        timeout=30
    )

    response.raise_for_status()

    soup = BeautifulSoup(
        response.text,
        "html.parser"
    )

    return soup.get_text(
        "\n",
        strip=True
    )

## Retrieve MapleStory Data

In [ ]:
MAPLE_URL = "https://www.digitaltq.com/maplestory-classes-jobs"

maplestory_data = fetch_website_contents(
    MAPLE_URL
)

print(
    f"Downloaded {len(maplestory_data):,} characters"
)

## Job Selection

In [ ]:
job_name = "Kain"

print(job_name)

## Brochure System Prompt

In [ ]:
brochure_system_prompt = """
You are a professional MapleStory class analyst.

Create a detailed brochure for a MapleStory job.

Sections:

# Job Name

## Overview

## Class Group

## Playstyle

## Strengths

## Weaknesses

## Mobility

## Bossing

## Training

## Ideal Players

## Final Verdict

Rules:

- Use ONLY the provided website information.
- Do not invent information.
- If information is unavailable, state that.
- Write in polished markdown.
"""

## User Prompt Builder

In [7]:
def get_brochure_prompt(
    job_name,
    website_text
):

    return f"""
Website information:

{website_text}

Question:

Create a complete brochure for the MapleStory job:

{job_name}

Use only the information available
in the website text.
"""

## Generate Brochure

In [8]:
from IPython.display import Markdown
from IPython.display import display

def create_brochure(job_name):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content": brochure_system_prompt
            },
            {
                "role": "user",
                "content": get_brochure_prompt(
                    job_name,
                    maplestory_data
                )
            }
        ]
    )

    brochure = (
        response
        .choices[0]
        .message
        .content
    )

    display(
        Markdown(
            brochure
        )
    )

    return brochure

## Create Brochure

In [ ]:
create_brochure(job_name)

## Example Workflow

## Optional Upgrade

### Recruit Recommendation Prompt
Add another feature:

In [10]:
recommendation_system_prompt = """
You are a MapleStory class advisor.

Based on the brochure,
recommend whether this class is suitable for:

- Beginner
- Intermediate
- Advanced

Explain why.
"""

### Recommendation Function

In [11]:
def recommend_player_type(brochure):

    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "system",
                "content":
                recommendation_system_prompt
            },
            {
                "role": "user",
                "content":
                brochure
            }
        ]
    )

    return (
        response
        .choices[0]
        .message
        .content
    )


## Create Brochure + Recommendation

In [ ]:
# Create Brochure + Recommendation

brochure = create_brochure(job_name)

recommendation = recommend_player_type(brochure)

display(Markdown(recommendation))

## Summary
Built an AI-powered MapleStory Character Brochure Generator using GPT-4.1-mini. The application retrieves MapleStory class information from external sources, generates structured brochures for players, and provides suitability recommendations through a multi-step LLM workflow inspired by retrieval-augmented generation (RAG) and agentic AI design patterns.